# DPO sharding bug explainer (expanded)

This notebook explains, with toy examples:
1. what a tracer is
2. what `jit` does in this context
3. what went wrong in sharding
4. why your workaround worked
5. how PR #2463 removed the need for the workaround

Related PRs:
- https://github.com/marin-community/marin/pull/2460
- https://github.com/marin-community/marin/pull/2463
- https://github.com/marin-community/marin/pull/2502

## Mental model

Think of JAX transformations (`jit`, `vmap`, `grad`) as running your Python function in two phases:

- Phase A: tracing phase
  - values are symbolic placeholders called *tracers*
  - operations are recorded into an intermediate program (jaxpr / lowered program)
- Phase B: execution phase
  - compiled program runs on real arrays

In tracing, some objects can temporarily look different from final runtime objects. That is exactly where this sharding issue came from.

## What is a tracer?

A tracer is a symbolic array-like object JAX uses during transformation.

It usually has:
- abstract shape and dtype
- transformation metadata (for example vmap batch information)
- no concrete host value in the usual sense

Why this matters here:
- `haliax.shard()` has different logic for traced arrays vs normal arrays
- if code takes the tracer path too early or too broadly, sharding constraints can be applied at the wrong time

In [ ]:
# Toy tracer model (not real JAX internals)
from dataclasses import dataclass

@dataclass
class FakeArray:
    shape: tuple
    tracer: bool = False
    batch_dim: int | None = None

@dataclass
class NamedArrayToy:
    array: FakeArray
    axis_names: tuple[str, ...]

def is_in_jit_toy():
    # Global context signal. In real code this can be true even when individual leaves differ.
    return True

def is_jit_tracer_toy(x: FakeArray):
    # Per-leaf signal, similar to _is_jit_tracer(named.array)
    return x.tracer

def pspec_for_axis(axis_names):
    # Toy PartitionSpec representation
    return tuple(axis_names)

def apply_sharding_constraint_toy(named: NamedArrayToy):
    pspec = pspec_for_axis(named.axis_names)
    # Toy invariant: partition spec rank must match concrete rank
    if len(pspec) != len(named.array.shape):
        raise ValueError(
            f"rank mismatch: pspec rank {len(pspec)} vs array rank {len(named.array.shape)}"
        )
    return f"ok: sharded with pspec={pspec}"

## What does `jit` affect here?

In `lib/haliax/src/haliax/partitioning.py`, `shard()` picks a path:
- tracer-style path (`with_sharding_constraint` / `reshard`)
- eager path (`jax.device_put`)

Historically the decision used a global context check (`is_in_jit()`), which can be too broad.

If a function is tracing under `jit`, global check may say "yes" for everything, even if what you need is a per-leaf decision.

### Eager path vs tracer-style path (concrete meaning)

- **Tracer-style path**: used when a leaf is a tracer. In real `partitioning.py` this goes through sharding constraints (`with_sharding_constraint` / `reshard`) while tracing.
- **Eager path**: used when a leaf is a concrete array. In real `partitioning.py` this uses `jax.device_put`, which places data now.

So this is not just "in jit vs not in jit". The better question is: *is this specific leaf a tracer right now?*

In [ ]:
def path_old(named: NamedArrayToy):
    return "tracer-style constraint path" if is_in_jit_toy() else "eager device_put path"

def path_new(named: NamedArrayToy):
    return "tracer-style constraint path" if is_jit_tracer_toy(named.array) else "eager device_put path"

concrete_leaf = NamedArrayToy(FakeArray(shape=(8, 16), tracer=False), ("In", "Out"))
traced_leaf = NamedArrayToy(FakeArray(shape=(8, 16), tracer=True), ("In", "Out"))

print("old decision on concrete leaf:", path_old(concrete_leaf))
print("old decision on traced leaf:", path_old(traced_leaf))
print("new decision on concrete leaf:", path_new(concrete_leaf))
print("new decision on traced leaf:", path_new(traced_leaf))

In [ ]:
def shard_old_global_jit(named: NamedArrayToy):
    # Old style decision
    if is_in_jit_toy():
        return apply_sharding_constraint_toy(named)
    else:
        return "device_put path"

# Simulate an in-flight vmapped internal:
# actual array has leading batch axis (size 4), but named axes have not been updated yet.
in_flight_vmapped = NamedArrayToy(
    array=FakeArray(shape=(4, 8, 16), tracer=True, batch_dim=0),
    axis_names=("In", "Out"),
)

try:
    print(shard_old_global_jit(in_flight_vmapped))
except Exception as e:
    print("old behavior failure:", e)

### Tuple clarification (`tuple(axis_names)` vs `(axis_names,)`)

Your question is exactly right to ask. In Python:
- `tuple(("B", "T", "H"))` -> `("B", "T", "H")` (flat length 3)
- `(("B", "T", "H"),)` -> one element, and that element is a tuple (length 1)

In this toy notebook, `pspec_for_axis(axis_names)` returns `tuple(axis_names)`, so it is flat.

The intentional failure case is different: we set `shape=(4, 8, 16)` but `axis_names=("In", "Out")`. That means rank 3 array vs rank 2 names.

In [ ]:
axis_names = ("B", "T", "H")
flat = tuple(axis_names)
nested = (axis_names,)
print("flat:", flat, "len=", len(flat))
print("nested:", nested, "len=", len(nested))

print("toy pspec from flat names:", pspec_for_axis(axis_names), "len=", len(pspec_for_axis(axis_names)))

good = NamedArrayToy(FakeArray(shape=(4, 8, 16), tracer=False), ("B", "T", "H"))
bad = NamedArrayToy(FakeArray(shape=(4, 8, 16), tracer=False), ("In", "Out"))
print("good rank check:", len(pspec_for_axis(good.axis_names)), "vs", len(good.array.shape))
print("bad rank check:", len(pspec_for_axis(bad.axis_names)), "vs", len(bad.array.shape))

## Your workaround in #2460

You added this guard in `partitioning.py`:
```python
if getattr(named.array, "batch_dim", None) is not None:
    return named
```

Interpretation:
- detect vmapped batched tracer internals
- skip sharding at that intermediate point
- let downstream code shard later when axes are stabilized

It is a practical local fix and matches the symptom.

In [ ]:
def shard_with_workaround(named: NamedArrayToy):
    if named.array.batch_dim is not None:
        return "skip for now (workaround)"

    if is_in_jit_toy():
        return apply_sharding_constraint_toy(named)
    return "device_put path"

print(shard_with_workaround(in_flight_vmapped))

## David's fix in #2463

PR: https://github.com/marin-community/marin/pull/2463

Two important changes:

1. Narrow sharding path selection
- from global `is_in_jit()` decision
- to leaf-level `_is_jit_tracer(named.array)` decision

2. Move sharding to a safer boundary in stacked init
- `haliax.auto_sharded(stacked)` after `haliax.vmap(module.init, Block)(...)`
- this happens once the batch axis is represented in named axes

Net effect:
- less need for ad hoc tracer/batch guards in `partitioning.py`
- sharding intent is explicit at the point where the stacked structure is finalized

In [ ]:
def shard_david_style(named: NamedArrayToy):
    # New decision style: inspect this leaf only
    if is_jit_tracer_toy(named.array):
        return apply_sharding_constraint_toy(named)
    else:
        return apply_sharding_constraint_toy(named)

# This is the post-vmap, stable object you now shard
post_vmap_stable = NamedArrayToy(
    array=FakeArray(shape=(4, 8, 16), tracer=False, batch_dim=None),
    axis_names=("Block", "In", "Out"),
)

print(shard_david_style(post_vmap_stable))

## Timeline comparison

Before:
1. enter transformed context (`jit` / `vmap`)
2. intermediate vmapped leaves appear (possible `batch_dim`)
3. sharding logic may run too early via broad jit check
4. workaround skips these leaves

After #2463:
1. transformed context still exists
2. per-leaf tracer detection avoids over-broad path selection
3. stacked object is explicitly `auto_sharded` after vmapped construction
4. less reliance on intermediate-shape heuristics

Companion hardening in #2502:
- https://github.com/marin-community/marin/pull/2502
- `pspec_for()` uses `axis_names`, improving behavior for None-backed named arrays

## Map to real files

- Workaround location on your branch: `lib/haliax/src/haliax/partitioning.py`
- Per-leaf tracer check helper: `lib/haliax/src/haliax/partitioning.py` (`_is_jit_tracer`)
- Safer sharding boundary: `lib/haliax/src/haliax/nn/scan.py` (`stacked = haliax.auto_sharded(stacked)`)

The next section includes an actual runtime `jit(vmap(...))` demo with tracer type prints.

## Real runtime mini-demo (`jit(vmap(...))`)

What to look for:
- trace-time prints show tracer-like types
- prints usually happen on first call (compile/trace), not every call
- named demo shows the wrapper (`NamedArray`) and underlying array type during tracing

If your local env is missing `jax` or `haliax`, this cell prints a skip message.

In [ ]:
try:
    import jax
    import jax.numpy as jnp
    import haliax as hax
except Exception as e:
    print("Skipping real demo; missing runtime dependency:", e)
else:
    def raw_traced_fn(x):
        print("trace-time raw arg type:", type(x))
        return x + 1

    raw_compiled = jax.jit(jax.vmap(raw_traced_fn))
    x = jnp.arange(6, dtype=jnp.float32).reshape(3, 2)
    y = raw_compiled(x)
    print("raw output shape:", y.shape)

    # Usually no re-trace for same shape/dtype signature.
    _ = raw_compiled(x)

    B = hax.Axis("B", 3)
    D = hax.Axis("D", 2)
    named_input = hax.named(x, (B, D))

    def named_traced_fn(v):
        print("trace-time named wrapper:", type(v), "underlying array:", type(v.array))
        return v + 1

    named_compiled = jax.jit(lambda n: hax.vmap(named_traced_fn, B)(n))
    named_out = named_compiled(named_input)
    print("named output axis names:", named_out.axis_names)
